# An-Ra V4 — core-vnext TPU training (post-20k path)
#
# Fail-closed training on verified token packs with a WSD LR schedule.
# Design doc: docs/planning/POST_20K_AGI_PATH.md
#
# Before Run All:
#   1. Accelerator -> TPU v5e-8, complete verification, restart if needed.
#   2. Attach TWO Kaggle datasets:
#        a) checkpoint dataset  -> contains anra-v4-current-full-resume.pt (step 20000+)
#        b) token-pack dataset  -> a .tar.gz containing manifest.json + train/*.npy
#   3. Edit CONFIG in cell 4 only.
#
# This notebook REFUSES to train on unverified data. If you see
# 'REFUSING TO TRAIN', the attached pack failed verification - do not bypass.


In [ ]:
# 1. TPU runtime preflight — fail closed.
import importlib.util, os, sys
os.environ['PJRT_DEVICE'] = 'TPU'
if importlib.util.find_spec('torch_xla') is None:
    raise RuntimeError('TPU not attached. Settings > Accelerator > TPU v5e-8, verify, restart, Run All.')
import torch
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.runtime as xr
except (ImportError, OSError) as exc:
    raise RuntimeError(f'Unusable XLA runtime: {exc}') from exc
if int(xr.global_device_count()) != 8:
    raise RuntimeError(f'Need all 8 cores of v5e-8, got {xr.global_device_count()}. Reconnect.')
print({'device': str(xm.xla_device()), 'cores': xr.global_device_count(), 'torch': torch.__version__})

In [ ]:
# 2. Clone the exact training branch and record its commit for provenance.
import json, os, subprocess, sys
from pathlib import Path
REPO_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
REPO_REF = 'core-vnext'
REPO = Path('/kaggle/working/anra')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', f'origin/{REPO_REF}'], check=True)
sys.path.insert(0, str(REPO))
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
os.environ['ANRA_SOURCE_COMMIT'] = SOURCE_COMMIT
print(json.dumps({'repo': str(REPO), 'commit': SOURCE_COMMIT[:12], 'branch': REPO_REF}))

In [ ]:
# 3. Locate checkpoint + token pack. EXPLICIT selection; no auto-pick.
import hashlib, json, tarfile
from pathlib import Path
INPUT_ROOT = Path('/kaggle/input')

# Operator: leave null to auto-select ONLY when exactly one candidate exists.
PREFERRED_CHECKPOINT_NAME = None   # e.g. 'anra-v4-current-full-resume.pt' (the step-20k anchor)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def safe_extract(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:gz') as bundle:
        root = destination.resolve()
        for member in bundle.getmembers():
            target = (destination / member.name).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f'unsafe archive member: {member.name}')
        bundle.extractall(destination)

def find_checkpoint():
    candidates = []
    for p in INPUT_ROOT.rglob('*.pt'):
        try:
            payload = torch.load(p, map_location='cpu', weights_only=True)
            step = int(payload.get('global_step', -1)) if isinstance(payload, dict) else -1
            del payload
        except Exception:
            continue
        if step >= 0:
            candidates.append((p, step))
    if not candidates:
        raise FileNotFoundError('No readable full-resume .pt checkpoint in /kaggle/input.')
    if PREFERRED_CHECKPOINT_NAME:
        matches = [c for c in candidates if c[0].name == PREFERRED_CHECKPOINT_NAME]
        if not matches:
            raise RuntimeError(f'PREFERRED_CHECKPOINT_NAME={PREFERRED_CHECKPOINT_NAME!r} not found among {[str(c[0]) for c in candidates]}')
        return matches[0]
    if len(candidates) > 1:
        # Step 30.4k measured WORSE than step 20k: never auto-pick highest.
        raise RuntimeError(
            'Multiple checkpoints present and none explicitly selected. '
            f'Candidates: {[(str(p.name), s) for p, s in candidates]}. '
            'Set PREFERRED_CHECKPOINT_NAME above (evidence: step-20000 is the recovery anchor).'
        )
    return candidates[0]

def find_pack():
    archives = sorted(INPUT_ROOT.rglob('*.tar.gz'))
    if not archives:
        raise FileNotFoundError('No *.tar.gz token pack found; plain text datasets are refused by design.')
    if len(archives) > 1:
        raise RuntimeError(f'Multiple pack archives present: {[str(a) for a in archives]}. Keep exactly one attached.')
    dest = Path('/kaggle/working/pack')
    safe_extract(archives[0], dest)
    if not (dest / 'manifest.json').is_file() or not (dest / 'train').is_dir():
        raise RuntimeError('Archive lacks manifest.json + train/*.npy. Build it with training/pack_verify.build_manifest.')
    return dest

CHECKPOINT, RESUME_STEP = find_checkpoint()
PACK_ROOT = find_pack()
print(json.dumps({'checkpoint': str(CHECKPOINT), 'resume_step': RESUME_STEP,
                  'checkpoint_sha256': sha256_file(CHECKPOINT)[:16], 'pack_root': str(PACK_ROOT)}, indent=2))

In [ ]:
# 4. CONFIG — edit only these values.
RESUME_FROM   = str(CHECKPOINT)      # step >= 20000 recommended (see POST_20K_AGI_PATH.md)
PACK_DIR      = str(PACK_ROOT)
OUTPUT_CKPT   = '/kaggle/working/anra-v4-trained.pt'
MAX_MINUTES   = 430                  # leave headroom before Kaggle's 9h limit
TOKEN_BUDGET  = 330_000_000          # unique tokens in THIS pack; decay lands at its end
BATCH_SIZE    = 1
GRAD_ACCUM    = 8                    # 1*8*8*2048 = 131,072 tokens/step
LEARNING_RATE = 2e-4                 # WSD: 2% warmup -> stable -> decay to 10%
MIN_LR_RATIO  = 0.10
SAVE_INTERVAL = 400
LOG_INTERVAL  = 10
SEED          = 1301
print(json.dumps({k: v for k, v in globals().items() if k.isupper()}, indent=2, default=str))

In [ ]:
# 5. Verify the pack FAIL-CLOSED before any GPU-hour is spent.
from pathlib import Path
from training.pack_verify import PackVerificationError, verify_pack
try:
    pack = verify_pack(Path(PACK_DIR))
except PackVerificationError as exc:
    raise SystemExit(f'REFUSING TO TRAIN: {exc}') from exc
print(f'PACK VERIFIED: {len(pack.shard_paths)} shards | '
      f'{pack.total_tokens:,} tokens | ~{pack.total_windows:,} unique windows')

In [ ]:
# 6. FAIL-FAST PREFLIGHT: identity, contract, pack semantics, resume semantics.
# Nothing expensive has started yet; any failure here stops Run All.
import json
from pathlib import Path
from anra_core.checkpoint import load_core_checkpoint
from training.pack_verify import PackVerificationError, verify_pack

# Checkpoint identity + tokenizer contract (legacy fallback is explicit, not silent).
try:
    model, payload, identity = load_core_checkpoint(str(CHECKPOINT))
    legacy = False
except Exception as exc:
    print(f'strict load failed ({type(exc).__name__}); explicit legacy load')
    model, payload, identity = load_core_checkpoint(str(CHECKPOINT), legacy_unverified=True)
    legacy = True
assert identity.tokenizer_contract_verified or legacy, 'tokenizer contract must be verified'
params = sum(p.numel() for p in model.parameters())
has_optimizer = bool(payload.get('optimizer') or payload.get('optimizer_state_dict'))

# Pack: hashes AND semantics (dtype, 1-D, token range, totals, block contract).
try:
    pack = verify_pack(Path(PACK_ROOT), vocab_size=32768, expected_block_size=2048)
except PackVerificationError as exc:
    raise SystemExit(f'REFUSING TO TRAIN: {exc}')

# Resume semantics preview: how many updates will this session actually run?
tokens_per_step = BATCH_SIZE * GRAD_ACCUM * 8 * pack.block_size
budget_steps = max(1, TOKEN_BUDGET // tokens_per_step)
restored_pack = int((payload.get('pack_step') or 0)) if isinstance(payload, dict) else 0
updates = max(0, budget_steps - restored_pack)
print(json.dumps({
    'resume_step': identity.global_step, 'legacy_load': legacy,
    'dense_params': params,
    'parameter_sha256': (identity.parameter_sha256 or '')[:16],
    'optimizer_state_in_artifact': has_optimizer,
    'pack_tokens': pack.total_tokens, 'pack_block': pack.block_size,
    'pack_windows': pack.total_windows,
    'tokens_per_step': tokens_per_step, 'pack_budget_steps': budget_steps,
    'restored_pack_step': restored_pack, 'updates_this_session': updates,
    'expected_workers': 8,
}, indent=2))
assert updates > 0, 'pack already consumed per its pack_step; attach a fresh pack'
if not has_optimizer and identity.global_step and identity.global_step > 0:
    print('NOTE: artifact has no optimizer state -> AdamW starts fresh (honest model_only-style resume).')
del model

In [ ]:
# 7. Train. WSD schedule decays LR across TOKEN_BUDGET so the run ends at
# the pack boundary instead of grinding repeat passes at full learning rate.
import sys
sys.path.insert(0, str(REPO))
os.chdir(str(REPO))

from training.train_tpu import parse_args, run

argv = [
    '--pack-root', PACK_DIR,
    '--resume-from', RESUME_FROM,
    '--output-checkpoint', OUTPUT_CKPT,
    '--max-minutes', str(MAX_MINUTES),
    '--token-budget', str(TOKEN_BUDGET),
    '--batch-size', str(BATCH_SIZE),
    '--grad-accum-steps', str(GRAD_ACCUM),
    '--learning-rate', str(LEARNING_RATE),
    '--min-lr-ratio', str(MIN_LR_RATIO),
    '--save-interval', str(SAVE_INTERVAL),
    '--log-interval', str(LOG_INTERVAL),
    '--seed', str(SEED),
]
exit_code = run(parse_args(argv))
print(f'training exit code: {exit_code}')
assert exit_code == 0, 'training failed - inspect log above'

In [ ]:
# 8. Post-session capability probe (v2) on CPU.
# CoreExecutor does not advertise XLA inference; correctness over convenience.
# Raw profile measures LEARNED behavior; assisted measures practical behavior.
import json
from connector.experiments.cognitive_credit.capability_probe import family_gates, run_probe
probe_raw = run_probe(OUTPUT_CKPT, device='cpu', profile='raw')
probe_assisted = run_probe(OUTPUT_CKPT, device='cpu', profile='assisted')
gates = family_gates(probe_raw)
print('RAW (learned behavior):')
print(json.dumps(probe_raw, indent=2))
print('ASSISTED (with decode controls):')
print(json.dumps(probe_assisted, indent=2))
print('FAMILY GATES (from raw):', json.dumps(gates, indent=2))
runnable = [f for f, ok in gates.items() if ok]
print('Runnable cognitive-credit families:', runnable or 'NONE - continue Phase B')

In [ ]:
# 9. Persist artifacts with full evidence identity (anra-evidence/v1).
import shutil, time
from evaluation.evidence import EvidenceIdentity, write_evidence
identity_record = EvidenceIdentity(
    source_commit=SOURCE_COMMIT,
    checkpoint_file_sha256=sha256_file(Path(OUTPUT_CKPT)),
    checkpoint_parameter_sha256='',  # filled by loader on next inspection
    global_step=identity.global_step + updates,
    tokenizer_identity='v4_32k',
    architecture_identity=str(identity.architecture_id),
    execution_profile='tpu_v5e8_bf16_wsd',
    decode_policy={'profile': 'raw+assisted recorded separately in probe cells'},
    supersedes=['ckpt_eval/step30400 (degraded: repeat passes at constant LR)'],
)
write_evidence(Path('/kaggle/working/session_evidence.json'), identity_record, {
    'probe_raw': probe_raw, 'probe_assisted': probe_assisted,
    'family_gates': gates, 'updates_this_session': updates,
    'pack_root': PACK_DIR,
})
print('Upload to the Drive vault:')
print(' ', OUTPUT_CKPT)
print('  /kaggle/working/session_evidence.json')